In this example, we will study the skyrmion dynamics in a 2d film. We will save the skyrmion
positions to a text file and and generate a movie at the end of simulation.

We import the necessary modules

In [1]:
using JuMag
using Printf

We use the double float precision in the simulation

In [2]:
JuMag.cuda_using_double(true)

The studied system is a 800nm x 300nm x 2nm film with periodic boundary conditions.

In [3]:
mesh =  FDMeshGPU(nx=400, ny=150, nz=1, dx=2e-9, dy=2e-9, dz=2e-9, pbc="xy")

FDMeshGPU{Float64}(2.0e-9, 2.0e-9, 2.0e-9, 400, 150, 1, 60000, true, true, false, 8.000000000000002e-27)

We create a Sim instance using `create_sim` function, and set basic simulation parameters such as
'Ms', 'A', 'D' and 'H'.

In [4]:
sim = create_sim(mesh,  Ms=3.87e5, A = 5.2e-12, D = 1e-3, H=(0,0,160*mT), name="skx")

JuMag.MicroSimGPU{Float64}(FDMeshGPU{Float64}(2.0e-9, 2.0e-9, 2.0e-9, 400, 150, 1, 60000, true, true, false, 8.000000000000002e-27), JuMag.EnergyMinimizationGPU{Float64}([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 0.0, 100.0, 1.0e-10, 0), DataSaver("skx_sd.txt", false, 0.0, 0, Any[SaverItem("step", "", JuMag.var"#67#69"()), SaverItem("E_total", "J", JuMag.var"#71#72"()), SaverItem(("m_x", "m_y", "m_z"), ("", "", ""), JuMag.average_m), SaverItem("E_exch", "J", JuMag.var"#179#180"{Int64}(1)), SaverItem("E_dmi", "J",

We set the initial magnetization configuration to a skyrmion at position (200nm, 80nm) with radius 20 nm.
Note that the initialized magnetization is a roughly guessing for the skyrimon.

In [5]:
init_m0_skyrmion(sim, (200e-9, 80e-9), 2e-8)

We relax the system to obtain the skyrmion profile.

In [6]:
relax(sim, maxsteps=20000, stopping_dmdt=0.01)

[ Info: Running Driver : JuMag.EnergyMinimizationGPU{Float64}.
[ Info: step =    1  step_size=9.049774e-16    max_dmdt=6.927667e+04
[ Info: step =    2  step_size=1.456321e-12    max_dmdt=5.196624e+04
[ Info: step =    3  step_size=4.298469e-13    max_dmdt=1.205125e+05
[ Info: step =    4  step_size=3.487176e-13    max_dmdt=5.211864e+04
[ Info: step =    5  step_size=2.402421e-13    max_dmdt=2.790084e+04
[ Info: step =    6  step_size=4.972522e-13    max_dmdt=1.253867e+04
[ Info: step =    7  step_size=6.851382e-13    max_dmdt=8.863601e+03
[ Info: step =    8  step_size=9.836863e-12    max_dmdt=7.017714e+03
[ Info: step =    9  step_size=2.616372e-13    max_dmdt=1.310473e+05
[ Info: step =   10  step_size=2.534689e-13    max_dmdt=5.802275e+04
[ Info: step =   11  step_size=2.256564e-13    max_dmdt=1.239998e+04
[ Info: step =   12  step_size=3.401806e-12    max_dmdt=9.467888e+03
[ Info: step =   13  step_size=1.750509e-12    max_dmdt=1.136784e+04
[ Info: step =   14  step_size=3.541412e

We save the magnetization to vtk, which can be opened using Paraview for 3D visualization.

In [7]:
JuMag.save_vtk(sim, "skx")

1-element Vector{String}:
 "skx.vts"

After obataining the skyrmion profile, we then move the skyrmion using spin transfer torques.
So we use change the driver to "LLG_STT" using the `set_driver` function. Meanwhile,
we set the relevant parameters such as alpha, beta and u.

In [8]:
set_driver(sim, driver="LLG_STT", alpha=0.05, beta=0.2, ux=-20)

During the simulation, we need to extract the skyrmion center, so we write a call back function
in which the skyrmion positions are computed using the `compute_guiding_center` function and
saved to a text file with append mode.

In [9]:
function call_back_fun(sim, t)
    Rx, Ry = compute_guiding_center(sim)
    open("XY.txt", "a") do f
        write(f, @sprintf("%g  %g  %g\n", t, Rx, Ry))
    end
end

call_back_fun (generic function with 1 method)

We use the `run_sim` function to run the simulation.
after that, a jdl2 file will be created and we can export it to a movie (.mp4) using the `jdl2movie`.

In [10]:
if !isfile("assets/skx.mp4")
    run_sim(sim, steps=100, dt=1e-10, save_m_every = 1, call_back=call_back_fun)
    jdl2movie("skx.jdl2", output="assets/skx.mp4")
end

![](./assets/skx.mp4)

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*